In [ ]:
import numpy as np
import mne
import scipy.io
import matplotlib.pyplot as plt
from train_utils import preprocess_eeg_raw # 假设train_utils中封装了基础转换

# 思路：首先从MAT文件中提取原始EEG和采样率，然后利用MNE构建Raw对象进行管道化处理。
# 特别注意：仅保留前64个电极，排除EOG等辅助通道。

def run_task1_preprocessing(file_path):
    # 1. 加载数据
    mat = scipy.io.loadmat(file_path)
    # 根据DTU-S2数据结构获取EEG (Time x Channels)
    eeg_data = mat['data']['eeg'] # 原始采样通常包含66+通道 [4]
    sfreq = 512.0 # DTU数据集标准采样率 [4]

    # 2. 提取前64个头皮电极并转置为 (Channels, Time)
    eeg_64 = eeg_data[:, :64].T
    ch_names = [f'EEG{i+1:03d}' for i in range(64)]
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
    raw = mne.io.RawArray(eeg_64, info)

    # 保存滤波前的状态用于对比可视化
    raw_raw = raw.copy()

    # 3. 滤波管道
    # FIR带通滤波 4-40Hz
    raw.filter(l_freq=4.0, h_freq=40.0, fir_design='firwin')
    # 50Hz陷波滤波
    raw.notch_filter(freqs=50.0)

    # 4. 可视化对比 (选择1秒片段，10个随机通道)
    start_idx = int(10 * sfreq) # 从第10秒开始
    stop_idx = int(11 * sfreq)
    picks = np.random.choice(64, 10, replace=False)

    data_before, times = raw_raw.get_data(picks=picks, start=start_idx, stop=stop_idx, return_times=True)
    data_after, _ = raw.get_data(picks=picks, start=start_idx, stop=stop_idx)

    plt.figure(figsize=(12, 6))
    plt.subplot(2, 1, 1)
    plt.plot(times, data_before.T)
    plt.title("Before Filtering (Raw 1s Segment)")
    plt.subplot(2, 1, 2)
    plt.plot(times, data_after.T)
    plt.title("After 4-40Hz Band-pass & 50Hz Notch")
    plt.tight_layout()
    plt.show()

    # 5. 重参考与重构
    raw.set_eeg_reference('average', projection=False)
    final_data = raw.get_data() # (64, Total_Samples)

    # 分段为 (N_segments, 64, 512)
    n_samples = final_data.shape
    n_segments = n_samples // 512
    eeg_reshaped = final_data[:, :n_segments*512].reshape(64, n_segments, 512)
    eeg_reshaped = np.transpose(eeg_reshaped, (1, 0, 2))

    return eeg_reshaped, raw.info

# 加载标签 (0=左, 1=右)
# labels = mat['expinfo']['attend_lr'] # 逻辑示意

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# 思路：
# 1. 将 (N, 64, 512) 展平为 (N, 32768)
# 2. 80/20 划分训练与测试集
# 3. 对比线性核与RBF核的性能

def run_task2_svm(X, y):
    n_samples = X.shape
    X_flat = X.reshape(n_samples, -1)

    # 划分数据集
    X_train, X_test, y_train, y_test = train_test_split(X_flat, y, test_size=0.2, random_state=42)

    # 标准化
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # 1. 线性SVM
    clf_lin = SVC(kernel='linear', C=1.0)
    clf_lin.fit(X_train, y_train)
    acc_lin = accuracy_score(y_test, clf_lin.predict(X_test))

    # 2. RBF核SVM (核探索)
    clf_rbf = SVC(kernel='rbf', C=1.0, gamma='scale')
    clf_rbf.fit(X_train, y_train)
    acc_rbf = accuracy_score(y_test, clf_rbf.predict(X_test))

    print(f"Linear SVM Accuracy: {acc_lin:.4f}")
    print(f"RBF SVM Accuracy: {acc_rbf:.4f}")
    return acc_lin, acc_rbf

In [ ]:
import torch
import torch.nn as nn
from train_utils import get_foundation_model, train_model # 模拟提供的工具库

# 思路：
# 1. 加载预训练的EEG基础模型 (如基于Transformer的架构)
# 2. 修改最后的全连接层以适应二分类 (左/右)
# 3. 设置极小的学习率进行微调

class AADFoundationModel(nn.Module):
    def __init__(self, pretrained_backbone):
        super().__init__()
        self.backbone = pretrained_backbone
        # 假设backbone输出维度为 256
        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)

def run_task3_fm(X, y, device='cuda'):
    # 转换为Tensor
    X_tensor = torch.FloatTensor(X).to(device) # (N, 64, 512)
    y_tensor = torch.LongTensor(y).to(device)

    # 加载模型
    backbone = get_foundation_model(model_name='eegformer_base')
    model = AADFoundationModel(backbone).to(device)

    # 仅微调分类头或设置分层学习率
    optimizer = torch.optim.AdamW([
        {'params': model.backbone.parameters(), 'lr': 1e-5},
        {'params': model.classifier.parameters(), 'lr': 1e-3}
    ], weight_decay=0.01)

    criterion = nn.CrossEntropyLoss()

    # 调用训练逻辑
    model = train_model(model, X_tensor, y_tensor, epochs=20, optimizer=optimizer, criterion=criterion)
    return model

In [ ]:
class ST_AADNet(nn.Module):
    def __init__(self, num_channels=64):
        super().__init__()
        # 1. 时间卷积: 学习不同频段的特征 [2]
        self.temp_conv = nn.Conv2d(1, 40, kernel_size=(1, 25), stride=(1, 1))
        # 2. 空间卷积: 跨通道整合信息
        self.spat_conv = nn.Conv2d(40, 30, kernel_size=(num_channels, 1))
        self.bn = nn.BatchNorm2d(30)
        self.pool = nn.AvgPool2d(kernel_size=(1, 20))

        # 3. LSTM层: 捕捉长程注意力动态
        self.lstm = nn.LSTM(input_size=30, hidden_size=30, batch_first=True)
        self.fc = nn.Linear(30, 2)

    def forward(self, x):
        # x: (Batch, Channels, Time) -> (B, 1, C, T)
        x = x.unsqueeze(1)
        x = torch.square(self.temp_conv(x)) # 非线性能量激活
        x = self.spat_conv(x)
        x = self.bn(x)
        x = self.pool(x)

        # 准备输入LSTM: (B, Time_Steps, Features)
        x = x.squeeze(2).permute(0, 2, 1)
        _, (hn, _) = self.lstm(x)
        return self.fc(hn.squeeze(0))